In [1]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class EvaluationScoreSchema(BaseModel):
    feedback: str = Field(description="Feedback on improving the essay")
    score: float = Field(description="Score of the essay from 0 to 10", ge=0, le=10)
    

In [7]:
llm_structured  = llm.with_structured_output(EvaluationScoreSchema)

In [8]:
#State Definition
import operator
from typing import Annotated


class EssayEvaluationState(TypedDict):
    essay: str
    language_feedback: str
    clarity_feedback: str
    factual_accuracy_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[float], operator.add, Field(description="List of individual scores for each criterion")]
    avg_score: float
    

In [9]:
from langchain_core.prompts import PromptTemplate

def evaluate_language(EssayEvaluationState: EssayEvaluationState) :
    prompt_template = PromptTemplate.from_template(
        template = "Act as a language expert and evaluate the language, grammar, fluency of the following essay and provide feedback and a score from 0 to 10:\n\n{essay}",
        input_variables = ["essay"],
        
    )
    chain = prompt_template | llm_structured
    language_evaluation = chain.invoke({"essay": EssayEvaluationState["essay"]})
    updated_state = { "language_feedback": language_evaluation['feedback'], "individual_scores": [language_evaluation['score']] } #type: ignore
    
    return updated_state

def evaluate_clarity(EssayEvaluationState: EssayEvaluationState) :
    prompt_template = PromptTemplate.from_template(
        template = "Act as a clarity expert and evaluate the clarity, coherence, and structure of the following essay and provide feedback and a score from 0 to 10:\n\n{essay}",
        input_variables = ["essay"],
        
    )
    chain = prompt_template | llm_structured
    clarity_evaluation = chain.invoke({"essay": EssayEvaluationState["essay"]})
    updated_state = { "clarity_feedback": clarity_evaluation['feedback'], "individual_scores": EssayEvaluationState["individual_scores"] + [clarity_evaluation['score']] } #type: ignore
    
    return updated_state

def evaluate_factual_accuracy(EssayEvaluationState: EssayEvaluationState) :
    prompt_template = PromptTemplate.from_template(
        template = "Act as a factual accuracy expert and evaluate the correctness of the mentioned facts and claims in the following essay and provide feedback and a score from 0 to 10:\n\n{essay}",
        input_variables = ["essay"],
        
    )
    chain = prompt_template | llm_structured
    factual_accuracy_evaluation = chain.invoke({"essay": EssayEvaluationState["essay"]})
    updated_state = { "factual_accuracy_feedback": factual_accuracy_evaluation['feedback'], "individual_scores": EssayEvaluationState["individual_scores"] + [factual_accuracy_evaluation['score']] } #type: ignore
    
    return updated_state

In [ ]:
graph = StateGraph(EssayEvaluationState)
